# maxpool-reduce — worked example 1: MaxPool2d as einops.reduce max

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `maxpool-reduce`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

Non-overlapping max pooling is a reduction over factored spatial axes. `einops.reduce(x, 'b c (h p1) (w p2) -> b c h w', 'max', p1=p, p2=p)` splits each spatial axis into a kept index and a window index, then takes the max across the window indices — reproducing `nn.MaxPool2d(p)` when stride equals kernel.

## Worked solution

We rebuild MaxPool2d from the reduce primitive.

1. **Factor the axes.** `(h p1)` tells einops the input height is `h * p1`; likewise width. With `p1 = p2 = p`, each `p x p` window is grouped.
2. **Reduce with 'max'.** The output pattern `b c h w` drops `p1` and `p2`, and `'max'` collapses them by taking the maximum — exactly the pooling operation.
3. **Pass factor sizes.** `p1=p, p2=p` lets einops solve the factorization; without them the split is ambiguous.
4. **Verify.** We compare against `F.max_pool2d(x, kernel_size=p)` which uses kernel = stride = p by default.

The demo pools a random `(2, 3, 4, 6)` tensor with `p=2`, prints the output shape `(2, 3, 2, 3)`, and confirms it matches torch.

In [ ]:
import torch as t
import torch.nn.functional as F
import einops

t.manual_seed(0)

def maxpool_via_reduce(x, p):
    return einops.reduce(x, 'b c (h p1) (w p2) -> b c h w', 'max', p1=p, p2=p)

x = t.randn(2, 3, 4, 6)
out = maxpool_via_reduce(x, 2)
ref = F.max_pool2d(x, kernel_size=2)
print('out shape:', tuple(out.shape))
print('matches F.max_pool2d:', t.allclose(out, ref))